# mamitas: foot segmentation with a MobileNetV3 U-Net (Kaggle)

Before running, check the notebook settings (right panel):
1. **Add Input** → dataset `lucasiturriago/feet-mamitas` → **Version 3**.
2. **Accelerator** → GPU (P100 or T4).
3. **Internet** → On (needed for `git clone` and the pretrained MobileNetV3 weights).

In [ ]:
!git clone https://github.com/andrearopero/mamitas-unet
%cd mamitas-unet
!pip install -q .

Locate the dataset folder (the one that contains `Train/`, `Valid/`, `Test/`).

In [ ]:
from pathlib import Path

roots = sorted({p.parent for p in Path("/kaggle/input").rglob("Train") if (p / "images").is_dir()})
print(roots)
assert roots, "Dataset not found: add feet-mamitas (v3) with 'Add Input'."
DATASET_DIR = str(roots[0])
for split in ("Train", "Valid", "Test"):
    print(split, len(list((Path(DATASET_DIR) / split / "images").glob("*"))), "images")

## Train
`configs/default.yaml` holds the paper's hyperparameters (Adam, batch 36, 60 epochs, ...).

In [ ]:
!python scripts/train.py --config configs/default.yaml --dataset-dir "{DATASET_DIR}"

## Predict on the Test split
Saves the masks and `metrics.json` in `runs/default/predictions/`.

In [ ]:
!python scripts/predict.py --config configs/default.yaml --dataset-dir "{DATASET_DIR}"

## Results

In [ ]:
import json
import matplotlib.pyplot as plt

history = json.load(open("runs/default/history.json"))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot([h["train_loss"] for h in history], label="train")
ax[0].plot([h["val_loss"] for h in history], label="valid")
ax[0].set_title("loss"); ax[0].legend()
ax[1].plot([h["val_dice"] for h in history], label="Dice")
ax[1].plot([h["val_iou"] for h in history], label="IoU")
ax[1].set_title("validation metrics"); ax[1].legend()
plt.show()
print(json.load(open("runs/default/predictions/metrics.json")))

In [ ]:
import numpy as np
from PIL import Image

root = Path(DATASET_DIR)
test = next(d for d in root.iterdir() if d.name.lower() == "test")
names = sorted(p.stem for p in (test / "images").glob("*"))[:4]
mask_dir = test / "masks" / "class_0" if (test / "masks" / "class_0").is_dir() else test / "masks"
fig, ax = plt.subplots(3, len(names), figsize=(3 * len(names), 9))
for j, n in enumerate(names):
    img = next((test / "images").glob(n + ".*"))
    ax[0, j].imshow(Image.open(img)); ax[0, j].set_title(n)
    ax[1, j].imshow(np.array(Image.open(mask_dir / (n + ".png"))) > 0, cmap="gray")
    ax[2, j].imshow(Image.open(f"runs/default/predictions/{n}.png"), cmap="gray")
for a in ax.ravel(): a.axis("off")
ax[0, 0].text(-0.1, 0.5, "image", transform=ax[0, 0].transAxes, rotation=90, va="center", ha="right")
ax[1, 0].text(-0.1, 0.5, "ground truth", transform=ax[1, 0].transAxes, rotation=90, va="center", ha="right")
ax[2, 0].text(-0.1, 0.5, "prediction", transform=ax[2, 0].transAxes, rotation=90, va="center", ha="right")
plt.show()